In [1]:
using Revise
using MSUC
using Random
using Plots
using Gurobi
using Statistics
const GRB_ENV = Gurobi.Env()
optimizer=() -> Gurobi.Optimizer(GRB_ENV)

Set parameter Username
Set parameter LicenseID to value 2658456
Academic license - for non-commercial use only - expires 2026-04-28


#11 (generic function with 1 method)

In [2]:
T = 24
graph=MSUC.LinearGraph(T+1);

In [11]:
instance=MSUC.parse_nc4("Data/T-Ramp/10_0_1_w.nc4",  optimizer, 24);1

1

In [ ]:
resSP=MSUC.benders_callback(instance, MSUC.extended_BD_FH_OH, silent=false, force=0.0, S=1, batch=1, gap=0.1, timelimit=120);

In [14]:
force = 1.0
model = MSUC.PolicyGraph(
    MSUC.subproblem_builder_UC_def,
    instance,
    force,
    graph,
    MSUC.ExtendedFormulation,
    sense = :Min,
    lower_bound = 0.0,
    optimizer = optimizer,
)

A policy graph with 25 nodes.
 Node indices: 1, ..., 25


In [ ]:
length(values(model[1].states))

Dict{Symbol, MSUC.State{VariableRef}} with 740 entries:
  Symbol("is_on[8,4]")        => State{VariableRef}(is_on[8,4]_in, is_on[8,4]_o…
  Symbol("start_up[3,13]")    => State{VariableRef}(start_up[3,13]_in, start_up…
  Symbol("start_down[7,11]")  => State{VariableRef}(start_down[7,11]_in, start_…
  Symbol("is_on[4,4]")        => State{VariableRef}(is_on[4,4]_in, is_on[4,4]_o…
  Symbol("start_up[6,1]")     => State{VariableRef}(start_up[6,1]_in, start_up[…
  Symbol("start_up[5,11]")    => State{VariableRef}(start_up[5,11]_in, start_up…
  Symbol("start_down[6,3]")   => State{VariableRef}(start_down[6,3]_in, start_d…
  Symbol("start_down[10,16]") => State{VariableRef}(start_down[10,16]_in, start…
  Symbol("is_on[3,5]")        => State{VariableRef}(is_on[3,5]_in, is_on[3,5]_o…
  Symbol("start_down[10,18]") => State{VariableRef}(start_down[10,18]_in, start…
  Symbol("start_up[10,18]")   => State{VariableRef}(start_up[10,18]_in, start_u…
  Symbol("start_up[7,16]")    => State{VariableRef}(s

In [16]:
MSUC.train(model; iteration_limit = 1)

-------------------------------------------------------------------
         MSUC.jl (c) Oscar Dowson and contributors, 2017-25
-------------------------------------------------------------------
problem
  nodes           : 25
  state variables : 740
  scenarios       : 1.00000e+00
  existing cuts   : false
options
  solver          : serial mode
  risk measure    : MSUC.Expectation()
  sampling scheme : MSUC.InSampleMonteCarlo
subproblem structure
  VariableRef                             : [3449, 3453]
  AffExpr in MOI.EqualTo{Float64}         : [995, 1451]
  AffExpr in MOI.GreaterThan{Float64}     : [1, 11]
  AffExpr in MOI.LessThan{Float64}        : [40, 960]
  VariableRef in MOI.EqualTo{Float64}     : [1, 1]
  VariableRef in MOI.GreaterThan{Float64} : [2708, 2710]
  VariableRef in MOI.LessThan{Float64}    : [740, 740]
  VariableRef in MOI.ZeroOne              : [730, 730]
numerical stability report
  matrix range     [1e+00, 1e+03]
  objective range  [1e+00, 3e+03]
  bounds range 

In [36]:
model[1].bellman_function.global_theta.cuts[1].constraint_ref

61211.39751433136 power[1]_out + 15828.956592000002 power[3]_out + 15706.967567999995 power[5]_out + 1273.832804 power[6]_out + 640.782625 power[7]_out + 94279.61420737603 power[9]_out + 1291.306312 power[10]_out - 63691.640199999994 is_on[6,0]_out - 39728.522750000004 is_on[7,0]_out - 108469.730208 is_on[10,0]_out + 4.529876509062255e6 is_on[1,1]_out + 1.0863777133380782e6 is_on[3,1]_out + 1.0869890988013744e6 is_on[5,1]_out + 1.6462939075136011e7 is_on[9,1]_out + 4.48126858504147e6 is_on[1,2]_out + 1.0411119752823251e6 is_on[3,2]_out + 1.0416978863513172e6 is_on[5,2]_out - 768443.3006799996 is_on[8,2]_out + 1.6351390904678935e7 is_on[9,2]_out + 4.432660661020685e6 is_on[1,3]_out + 984433.9702902733 is_on[2,3]_out + 3.7117905205717674e6 is_on[3,3]_out + 9.022318770791132e6 is_on[4,3]_out + 2.853346384353608e6 is_on[5,3]_out + 1.6645819403222867e6 is_on[8,3]_out + 1.6239842734221859e7 is_on[9,3]_out + 4.3840527369999e6 is_on[1,4]_out + 939686.9716407154 is_on[2,4]_out + 8.9729286053469

In [25]:
using JuMP
using Gurobi

# --- Création du modèle ---
model = Model(Gurobi.Optimizer)

# --- Variables ---
@variable(model, x >= 0)          # quantité du produit A
@variable(model, y >= 0)          # quantité du produit B
@variable(model, z<=1)          # 1 si la machine est allumée

JuMP.fix(z, 1.0; force = true)  # forcer la machine à être allumée

# --- Données ---
profit_A = 3.0
profit_B = 5.0
hours_A  = 1.0
hours_B  = 2.0
capacity = 6.0
bigM     = 6.0   # permet de lier la contrainte à la variable binaire

# --- Contraintes ---
@constraint(model, hours_A * x + hours_B * y <= capacity * z)
@constraint(model, x <= 4)    # limite sur la production de A
@constraint(model, y <= 3)    # limite sur la production de B

# --- Objectif ---
@objective(model, Max, profit_A * x + profit_B * y*z - 2 * z)  # -2 = coût d’activation

# --- Résolution ---
optimize!(model)

# --- Résultats ---
println("x = ", value(x))
println("y = ", value(y))
println("z = ", value(z))
println("Profit = ", objective_value(model))


Set parameter Username
Set parameter LicenseID to value 2658456
Academic license - for non-commercial use only - expires 2026-04-28
Gurobi Optimizer version 12.0.3 build v12.0.3rc0 (linux64 - "Ubuntu 22.04.5 LTS")

CPU model: Intel(R) Core(TM) Ultra 9 185H, instruction set [SSE2|AVX|AVX2]
Thread count: 22 physical cores, 22 logical processors, using up to 22 threads

Optimize a model with 3 rows, 3 columns and 5 nonzeros
Model fingerprint: 0x10b66120
Model has 1 quadratic objective term
Coefficient statistics:
  Matrix range     [1e+00, 6e+00]
  Objective range  [2e+00, 3e+00]
  QObjective range [1e+01, 1e+01]
  Bounds range     [1e+00, 1e+00]
  RHS range        [3e+00, 4e+00]
Presolve removed 2 rows and 1 columns
Presolve time: 0.00s
Presolved: 1 rows, 2 columns, 2 nonzeros
Ordering time: 0.00s

Barrier statistics:
 AA' NZ     : 0.000e+00
 Factor NZ  : 1.000e+00
 Factor Ops : 1.000e+00 (less than 1 second per iteration)
 Threads    : 1

                  Objective                Resid

In [23]:
println(model)

Max 5 z*y + 3 x - 2 z
Subject to
 x + 2 y - 6 z ≤ 0
 x ≤ 4
 y ≤ 3
 z = 1
 x ≥ 0
 y ≥ 0



In [24]:
optimize!(model)

Gurobi Optimizer version 12.0.3 build v12.0.3rc0 (linux64 - "Ubuntu 22.04.5 LTS")

CPU model: Intel(R) Core(TM) Ultra 9 185H, instruction set [SSE2|AVX|AVX2]
Thread count: 22 physical cores, 22 logical processors, using up to 22 threads

Optimize a model with 3 rows, 3 columns and 5 nonzeros
Model fingerprint: 0x10b66120
Model has 1 quadratic objective term
Coefficient statistics:
  Matrix range     [1e+00, 6e+00]
  Objective range  [2e+00, 3e+00]
  QObjective range [1e+01, 1e+01]
  Bounds range     [1e+00, 1e+00]
  RHS range        [3e+00, 4e+00]
Presolve removed 2 rows and 1 columns
Presolve time: 0.00s
Presolved: 1 rows, 2 columns, 2 nonzeros
Ordering time: 0.00s

Barrier statistics:
 AA' NZ     : 0.000e+00
 Factor NZ  : 1.000e+00
 Factor Ops : 1.000e+00 (less than 1 second per iteration)
 Threads    : 1

                  Objective                Residual
Iter       Primal          Dual         Primal    Dual     Compl     Time
   0   2.24280436e+01  2.02595226e+01  3.28e+00 1.80e+